In [ ]:
# 绘制三张图。0412最终代码


In [2]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. 🌟 自定义参数设定：专属 vmax
# ==========================================
vmax_lineage    = 0.01      # 第 1 张图：Lineage
vmax_functional = 0.0075    # 第 2 张图：Functional
vmax_noise      = 0.005     # 第 3 张图：Doublet_and_Artifact

# 配置文件路径
marker_info_path = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.2.cNMF_CD8T/3.5.CD8T.QC.MarkerGene/3.5.CD8T.cGEP.gene.csv'
spectra_score_path = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.2.cNMF_CD8T/Example_refbuilder_95_v2/starcat_refstarcat_consensus_spectra_score.filtered.txt'
out_dir = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.2.cNMF_CD8T/3.5.CD8T.QC.MarkerGene'

vmax_dict = {
    'Lineage': vmax_lineage,
    'Functional': vmax_functional,
    'Doublet_and_Artifact': vmax_noise
}

# ==========================================
# 2. 读取数据与基础排序
# ==========================================
print("正在读取数据...")
marker_df = pd.read_csv(marker_info_path)
spectra_df = pd.read_csv(spectra_score_path, sep='\t', index_col=0)

marker_df = marker_df.dropna(subset=['Class'])

custom_class_order = ['Lineage', 'Functional', 'Doublet', 'Artifact']
marker_df['Class'] = pd.Categorical(marker_df['Class'], categories=custom_class_order, ordered=True)

# ==========================================
# 3. 智能分组
# ==========================================
def assign_plot_group(c):
    c_str = str(c).strip()
    if c_str in ['Doublet', 'Artifact']:
        return 'Doublet_and_Artifact'
    return c_str

marker_df['Plot_Group'] = marker_df['Class'].apply(assign_plot_group)
unique_groups = marker_df['Plot_Group'].dropna().unique()

print(f"检测到 {len(unique_groups)} 个绘图分组: {list(unique_groups)}")

# ==========================================
# 4. 循环生成独立热图
# ==========================================
for current_group in unique_groups:
    
    current_vmax = vmax_dict.get(current_group, 0.0075)
    print(f"\n[{current_group}] 正在绘图，当前应用阈值 vmax = {current_vmax}")
    
    group_df = marker_df[marker_df['Plot_Group'] == current_group].copy()
    group_df['sort_key'] = group_df['Cluster'].astype(str).str.extract(r'(\d+)').astype(float)
    group_df = group_df.sort_values(by=['Class', 'sort_key'])
    
    ordered_genes = []
    toshow_main = []
    ytick_labels = []
    gene_counts_per_cluster = []
    
    class_boundaries = []        
    last_class = None
    current_row_idx = 0
    
    for _, row in group_df.iterrows():
        cluster_id = row['Cluster']
        anno_name = row['Annotation']
        current_class = row['Class']
        
        if cluster_id not in spectra_df.index:
            continue
            
        if last_class is not None and current_class != last_class:
            class_boundaries.append(current_row_idx)
        last_class = current_class
            
        toshow_main.append(cluster_id)
        ytick_labels.append(f"{cluster_id}: {anno_name}")
        
        raw_genes = [g.strip() for g in str(row['MarkerGene']).replace(',', ' ').split() if g.strip()]
        
        valid_unique_genes_for_this_cgep = []
        for g in raw_genes:
            if g in spectra_df.columns and g not in valid_unique_genes_for_this_cgep:
                valid_unique_genes_for_this_cgep.append(g)
                ordered_genes.append(g)
        
        gene_counts_per_cluster.append(len(valid_unique_genes_for_this_cgep))
        current_row_idx += 1
        
    if not ordered_genes:
        print(f"⚠️ {current_group} 没有提取到有效的特异性基因，已跳过。")
        continue

    Z = spectra_df.loc[toshow_main, ordered_genes].copy()
    Z[Z > current_vmax] = current_vmax
    Z = Z.fillna(0)

    width = max(6.0, len(ordered_genes) * 0.22)
    height = max(3.5, len(toshow_main) * 0.4) 
    
    fig, ax = plt.subplots(figsize=(width, height), dpi=300)
    plt.subplots_adjust(left=0.4, bottom=0.35, right=0.95, top=0.85)

    if current_group == 'Functional':
        cbar_offset = -0.23
    elif current_group == 'Lineage':
        cbar_offset = -0.28
    else:
        cbar_offset = -0.28
        
    cbar_ax = ax.inset_axes([0, cbar_offset, 0.15, 0.03], transform=ax.transAxes)

    sns.heatmap(
        Z, ax=ax, vmin=0, vmax=current_vmax, cmap='mako',                
        linewidths=0, cbar_ax=cbar_ax, 
        cbar_kws={"orientation": "horizontal", 'ticks': [0, current_vmax]}, 
        xticklabels=True, yticklabels=True
    )

    ax.hlines(y=np.arange(1, len(toshow_main)), xmin=0, xmax=len(ordered_genes), 
              colors='white', linewidth=0.5, alpha=0.5)
              
    if class_boundaries:
        ax.hlines(y=class_boundaries, xmin=0, xmax=len(ordered_genes), 
                  colors='white', linewidth=2.0)

    current_x = 0
    vlines_x = []
    for count in gene_counts_per_cluster:
        current_x += count
        if count > 0: vlines_x.append(current_x)
    if vlines_x: ax.vlines(x=vlines_x[:-1], ymin=0, ymax=len(toshow_main), 
                           colors='white', linewidth=0.6, alpha=0.7)

    for _, spine in ax.spines.items():
        spine.set_visible(True); spine.set_color('white'); spine.set_linewidth(1.5)

    # ==========================================
    # 🌟 坐标轴与表头文字排版 (更新部分)
    # ==========================================
    # Y轴设置
    ax.set_yticks(np.arange(len(toshow_main)) + 0.5)
    ax.set_yticklabels(ytick_labels, fontsize=13, rotation=0)
    ax.set_ylabel('cGEP', fontsize=14, labelpad=10)  # Y轴名称

    # X轴设置
    ax.set_xticks(np.arange(len(ordered_genes)) + 0.5)
    ax.set_xticklabels(ordered_genes, fontsize=9, rotation=90) 
    ax.set_xlabel('Marker', fontsize=14, labelpad=10)  #  X轴名称
    
    # 表头修改：结合细胞类型 (current_group) 
    ax.set_title(f'{current_group} cGEP Example marker genes/proteins', fontsize=15, pad=20)
    
    # 色标设置
    cbar_ax.set_xticklabels(['0', f'{current_vmax}'])
    cbar_ax.set_title('Spectra Score', fontsize=10, pad=5)

    # 保存
    save_path = os.path.join(out_dir, f'3.5.CD8T_Markers_{current_group}.png')
    plt.savefig(save_path, bbox_inches="tight")
    print(f"✅ 生成成功 -> {save_path}")
    
    plt.close(fig)

print("\n🎉 表头与坐标轴名称已全部更新完毕！")

正在读取数据...
检测到 3 个绘图分组: ['Functional', 'Lineage', 'Doublet_and_Artifact']

[Functional] 正在绘图，当前应用阈值 vmax = 0.0075
✅ 生成成功 -> /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.2.cNMF_CD8T/3.5.CD8T.QC.MarkerGene/3.5.CD8T_Markers_Functional.png

[Lineage] 正在绘图，当前应用阈值 vmax = 0.01
✅ 生成成功 -> /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.2.cNMF_CD8T/3.5.CD8T.QC.MarkerGene/3.5.CD8T_Markers_Lineage.png

[Doublet_and_Artifact] 正在绘图，当前应用阈值 vmax = 0.005
✅ 生成成功 -> /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.2.cNMF_CD8T/3.5.CD8T.QC.MarkerGene/3.5.CD8T_Markers_Doublet_and_Artifact.png

🎉 表头与坐标轴名称已全部更新完毕！
